# CLEF CheckThat! 2026 Task 2 (English): 10-config MLP ablation on the **top-5 ranked traces**
### Qwen3-Embedding-8B · p/n trace ranking (CV-selected blocks) · rank-aware attention · Kaggle 2×T4

Same pipeline, model, training and evaluation as `mlp-english-2.ipynb`, with one change in the input:
instead of all 20 reasoning traces, every claim is represented by the **5 traces ranked highest by our
ranking pipeline** (`pn_selected`: p/n rule, feature blocks chosen by 5-fold CV).

`Claim + top-5 traces (Qwen3-Embedding-8B, existing .npz) → [x_claim ; x_attention ; x_disagreement ; x_verdict]`
`→ BatchNorm → [Linear→BN→ReLU→Dropout]×3 (512/256/128) → 3-way → Softmax`, class-weighted cross-entropy.

**What changed compared with `mlp-english-2`**

| Part | `mlp-english-2` | This notebook |
|:--|:--|:--|
| Traces fed to the model | all traces (≤ 20) | the 5 highest-ranked by `pn_selected` |
| Attention | additive attention over 20 traces | **rank-aware**: a learnable rank embedding (rank 1..5) is added to each trace before scoring; the pooled vector is still a weighted sum of the trace embeddings. Initialised to zero, so training starts from the plain attention |
| Disagreement | masked std over 20 traces | masked std over the 5 traces (same module) |
| Verdict features (7) | from all 20 verdicts | from the 5 top-ranked traces' verdicts (`num_traces_norm` = valid votes / 5) |
| Embeddings | computed in the notebook | **loaded** from the existing `{train,val,test}_qwen3emb8b_en_v1.npz` (no 8B model, no re-embedding) |

Everything else is unchanged: 10 configurations, AE pre-training, optimiser, scheduler, early stopping on
validation loss, seed 42, metrics, plots and the LaTeX table.

**Which ranking each split uses (no label leakage).** *Test:* the ranker fitted on train+val. *Train / val:*
the out-of-fold rankings (5-fold over train+val), so no training row's top 5 was chosen by a model that saw its
label. No deduplication: all 6400 / 1600 / 2558 rows are used.

**Run on Kaggle:** accelerator **GPU T4 ×2**, internet not needed. Attach three inputs: the CheckThat! data,
the English embeddings and the output of `saving-ranked-english-traces` (folder `en_trace_rankings`). Check the
paths in §2, then *Save & Run All*. Finished experiments are skipped on re-runs (`FORCE_RERUN_EXPERIMENTS=False`).

## 1. Environment Setup

In [ ]:
import os, gc, re, json, time, glob, math, copy, random, warnings
from collections import Counter, OrderedDict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from sklearn.metrics import (confusion_matrix, accuracy_score, f1_score,
                             precision_score, recall_score)
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)

print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
for _i in range(torch.cuda.device_count()):
    _p = torch.cuda.get_device_properties(_i)
    print(f"  GPU {_i}: {_p.name}  {_p.total_memory/1e9:.1f} GB")

## 2. Configuration
Only the three input paths should need changing. Each is searched recursively, so pointing at a parent folder works.

In [ ]:
# ─────────────────────────── INPUT PATHS ───────────────────────────
DATA_DIR  = "/kaggle/input/datasets/d202511054/clef-2026-checkthat-task-2-data/English"          # train.json, validation.json, test.json
EMB_DIR   = "/kaggle/input/datasets/d202511054/qwen3-en-embeddings/qwen3 english embeddings"      # {train,val,test}_qwen3emb8b_en_v1.npz
RANK_DIR  = "/kaggle/input"                     # searched for en_trace_rankings/rankings/<RANK_METHOD>/{train,val,test}.json
OUTPUT_DIR = "/kaggle/working"

FORCE_RERUN_EXPERIMENTS = False                 # False = skip experiments whose result JSON already exists

# ─────────────────────────── TOP-K TRACES ───────────────────────────
RANK_METHOD = "pn_selected"                     # p/n, counts + CV-selected blocks (ours)
TOP_K       = 5

# ─────────────────────────── DATA FIELDS ───────────────────────────
CLAIM_FIELD        = "claim"
TRACE_TEXT_FIELD   = "Reasoning_traces"
VERDICT_LIST_FIELD = "Verdict_list"
LABEL_KEYS         = ("label", "Label")         # train/val use "label", test uses "Label"
TEST_ID_FIELD      = "query_id"
EMB_TAG            = "qwen3emb8b_en_v1"

label_map         = {"False": 0, "True": 1, "Conflicting": 2}
reverse_label_map = {v: k for k, v in label_map.items()}
CLASS_NAMES       = [reverse_label_map[i] for i in range(3)]

# ─────────────────────────── FEATURES / MODEL ───────────────────────────
SCALAR_DIM          = 7
EXPANDED_SCALAR_DIM = 1024
ATTN_HIDDEN         = 256
PROJ_HIDDEN         = 256
AE_HIDDEN           = 256
HIDDEN_DIMS         = [512, 256, 128]
DROPOUT             = 0.3
OUTPUT_DIM          = 3

# ─────────────────────────── MLP TRAINING ───────────────────────────
BATCH_SIZE          = 16
MAX_EPOCHS          = 100
LEARNING_RATE       = 1e-3
WEIGHT_DECAY        = 1e-5
GRAD_CLIP_NORM      = 1.0
EARLY_STOP_PATIENCE = 20
LR_PLATEAU_FACTOR   = 0.5
LR_PLATEAU_PATIENCE = 3
USE_AMP             = True

# ─────────────────────────── AUTOENCODER PRE-TRAINING ───────────────────────────
AE_EPOCHS     = 200
AE_LR         = 1e-3
AE_BATCH_SIZE = 64

RANDOM_SEED = 42
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
N_GPUS      = torch.cuda.device_count()

# ─────────────────────────── OUTPUT LAYOUT ───────────────────────────
RESULTS_DIR = f"{OUTPUT_DIR}/results"
MODELS_DIR  = f"{OUTPUT_DIR}/models"
PLOTS_DIR   = f"{OUTPUT_DIR}/plots"
PREDS_DIR   = f"{OUTPUT_DIR}/predictions"
for _d in [OUTPUT_DIR, RESULTS_DIR, MODELS_DIR, PLOTS_DIR, PREDS_DIR]:
    os.makedirs(_d, exist_ok=True)
print(f"Device: {DEVICE} ({N_GPUS} GPU) | top-{TOP_K} traces from '{RANK_METHOD}'")

## 3. Reproducibility

In [ ]:
def set_seed(seed: int = RANDOM_SEED) -> None:
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

set_seed()

def to_py(obj):
    if isinstance(obj, np.integer):  return int(obj)
    if isinstance(obj, np.floating): return float(obj)
    if isinstance(obj, np.ndarray):  return obj.tolist()
    if isinstance(obj, dict):        return {str(k): to_py(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)): return [to_py(v) for v in obj]
    return obj

with open(f"{OUTPUT_DIR}/experiment_meta.json", "w", encoding="utf-8") as _f:
    json.dump(to_py({k: v for k, v in globals().items()
                     if k.isupper() and isinstance(v, (int, float, str, bool, list, dict, type(None)))}),
              _f, indent=2, ensure_ascii=False)
print(f"Seed {RANDOM_SEED} | config saved → {OUTPUT_DIR}/experiment_meta.json")

## 4. Input Files, Dataset Loading & Schema Validation
IDs: train and validation use the position in the file (0..N-1); test uses `query_id` (the same keys as the
embedding files). The run stops with a clear error if a record is malformed.

In [ ]:
def find_one(root, rel):
    # rel = file name or a path tail like "rankings/pn_selected/train.json"; searched recursively under root
    if not os.path.isdir(root):
        raise FileNotFoundError(f"{root} does not exist — attach the input and fix the path in §2")
    hits = [h for h in glob.glob(os.path.join(root, "**", rel), recursive=True)
            if "arab" not in h.lower() and "span" not in h.lower()]
    hits.sort(key=len)
    if not hits:
        raise FileNotFoundError(f"'{rel}' not found under {root} — fix the path in §2")
    if len(hits) > 1:
        print(f"  note: {len(hits)} matches for {rel}; using the shortest path")
    return hits[0]

JSON_PATHS = {"train": find_one(DATA_DIR, "train.json"), "val": find_one(DATA_DIR, "validation.json"),
              "test": find_one(DATA_DIR, "test.json")}
EMB_PATHS  = {s: find_one(EMB_DIR, f"{s}_{EMB_TAG}.npz") for s in JSON_PATHS}
RANK_PATHS = {s: find_one(RANK_DIR, f"rankings/{RANK_METHOD}/{s}.json") for s in JSON_PATHS}
for s in JSON_PATHS:
    print(f"{s:5s} json   : {JSON_PATHS[s]}\n      emb    : {EMB_PATHS[s]}  ({os.path.getsize(EMB_PATHS[s]) / 1e9:.2f} GB)"
          f"\n      ranking: {RANK_PATHS[s]}")

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        s = f.read()
    try:
        return json.loads(s)
    except json.JSONDecodeError:
        # some copies of English train.json have one entry whose '"Reasoning_traces": [' key is missing
        print(f"  {os.path.basename(path)}: invalid JSON, repairing the missing 'Reasoning_traces' key")
        return json.loads(re.sub(r'\{\s*\nx?(\s+)"(?![A-Za-z_]+"\s*:)', r'{\n        "Reasoning_traces": [\n\1"', s))

def get_label(item):
    for k in LABEL_KEYS:
        if k in item:
            v = item[k]
            if v not in label_map:
                raise ValueError(f"Unknown label {v!r} (expected one of {list(label_map)})")
            return label_map[v]
    raise KeyError(f"No label key {LABEL_KEYS} in item with keys {list(item.keys())}")

def get_trace_texts(item):
    # Non-empty reasoning traces, original order (= rows 1.. of the embedding array).
    return [t for t in item[TRACE_TEXT_FIELD] if t.strip()]

def validate_items(items, split):
    errors = []
    for i, it in enumerate(items):
        if not isinstance(it.get(CLAIM_FIELD), str):
            errors.append(f"[{split}#{i}] '{CLAIM_FIELD}' missing or not a string")
        tr = it.get(TRACE_TEXT_FIELD)
        if not isinstance(tr, list) or not all(isinstance(t, str) for t in tr):
            errors.append(f"[{split}#{i}] '{TRACE_TEXT_FIELD}' must be a list of strings, got {type(tr).__name__}")
        vl = it.get(VERDICT_LIST_FIELD)
        if not isinstance(vl, list):
            errors.append(f"[{split}#{i}] '{VERDICT_LIST_FIELD}' must be a list, got {type(vl).__name__}")
        try: get_label(it)
        except Exception as e: errors.append(f"[{split}#{i}] {e}")
        if len(errors) >= 20: break
    if errors:
        raise ValueError(f"{split}: schema errors (first {len(errors)}):\n  " + "\n  ".join(errors))

def build_split(items, split, id_mode):
    validate_items(items, split)
    if id_mode == "index":
        ids = list(range(len(items)))
    elif id_mode == "query_id":
        ids = [it[TEST_ID_FIELD] for it in items]
        if len(set(ids)) != len(ids):
            raise ValueError(f"{split}: duplicate {TEST_ID_FIELD} values")
    else:
        raise ValueError(id_mode)
    return {"name": split, "items": items, "ids": ids,
            "id_to_item": dict(zip(ids, items)), "order": sorted(ids)}

SPLITS = OrderedDict()
SPLITS["train"] = build_split(load_json(JSON_PATHS["train"]), "train", "index")
SPLITS["val"]   = build_split(load_json(JSON_PATHS["val"]),   "val",   "index")
SPLITS["test"]  = build_split(load_json(JSON_PATHS["test"]),  "test",  "query_id")
for s, d in SPLITS.items():
    first = d["id_to_item"][d["order"][0]]
    print(f"{s:<5}: {len(d['order']):>6} items | keys={list(first.keys())}")
print("✓ Schema valid for all splits")

## 5. Top-K Trace Selection from the Ranking Files
Each ranking file holds one record per row with `order` = trace indices, best first. The records are matched
to the data by position (train/val) or `query_id` (test), and the claim text and label are checked. The first
`TOP_K` ranked traces that have an embedding are selected (a trace without text has no embedding row).

In [ ]:
def load_top_k(split):
    d = SPLITS[split]
    recs = load_json(RANK_PATHS[split])
    if len(recs) != len(d["items"]):
        raise ValueError(f"{split}: ranking file has {len(recs)} rows, the data has {len(d['items'])} — "
                         f"wrong ranking file?")
    key = (lambda r: r["query_id"]) if split == "test" else (lambda r: r["row"])
    by_id = {key(r): r for r in recs}
    top, full_recall, n_skipped = {}, [], 0
    for id_ in d["order"]:
        it, r = d["id_to_item"][id_], by_id.get(id_)
        if r is None:
            raise KeyError(f"{split}: no ranking record for id {id_}")
        if r["claim"] != it[CLAIM_FIELD] or r["label"] != it.get("label", it.get("Label")):
            raise ValueError(f"{split} id {id_}: ranking record does not match the data (claim/label differ)")
        if sorted(r["order"]) != list(range(len(it[VERDICT_LIST_FIELD]))):
            raise ValueError(f"{split} id {id_}: 'order' is not a permutation of the traces")
        has_emb = {j for j, t in enumerate(it[TRACE_TEXT_FIELD]) if t.strip()}
        picked = [j for j in r["order"] if j in has_emb][:TOP_K]
        n_skipped += sum(j not in has_emb for j in r["order"][:TOP_K])
        top[id_] = picked
        full_recall.append(r.get(f"recall@{TOP_K}", np.nan))
    return top, float(np.nanmean(full_recall)), n_skipped

TOPK, rank_rows = {}, {}
for s, d in SPLITS.items():
    TOPK[s], rec, skipped = load_top_k(s)
    items = [d["id_to_item"][i] for i in d["order"]]
    labels = [it.get("label", it.get("Label")) for it in items]
    rel = [np.mean([it[VERDICT_LIST_FIELD][j] == y for j in TOPK[s][i]]) if TOPK[s][i] else 0.0
           for i, it, y in zip(d["order"], items, labels)]
    rank_rows[s] = {"rows": len(items), f"traces fed (mean)": np.mean([len(TOPK[s][i]) for i in d["order"]]),
                    f"rows with < {TOP_K} traces": sum(len(TOPK[s][i]) < TOP_K for i in d["order"]),
                    "ranked traces skipped (no embedding)": skipped,
                    f"Recall@{TOP_K} of the ranking": round(rec, 4),
                    f"share of top-{TOP_K} matching gold": round(float(np.mean(rel)), 4)}
display(pd.DataFrame(rank_rows))

## 6. Data Diagnostics
Label balance, verdict values, and a *diagnostic-only* reference: the plurality vote (ties → Conflicting) over
**all** verdicts and over the **top-5** verdicts that the model now sees.

In [ ]:
def plurality(votes):
    c = Counter(label_map[v] for v in votes if v in label_map)
    if not c: return 2
    top = c.most_common(); best = top[0][1]
    winners = [k for k, v in top if v == best]
    return winners[0] if len(winners) == 1 else 2

def data_diagnostics(split):
    d = SPLITS[split]
    items  = [d["id_to_item"][i] for i in d["order"]]
    vals   = Counter(v for it in items for v in it[VERDICT_LIST_FIELD])
    labels = np.array([get_label(it) for it in items])
    p_all  = np.array([plurality(it[VERDICT_LIST_FIELD]) for it in items])
    p_top  = np.array([plurality([it[VERDICT_LIST_FIELD][j] for j in TOPK[split][i]]) for i, it in zip(d["order"], items)])
    f1 = lambda p: round(f1_score(labels, p, average="macro", labels=[0, 1, 2], zero_division=0), 4)
    return {"items": len(items),
            "traces min/mean/max": f"{min(len(it[TRACE_TEXT_FIELD]) for it in items)}/"
                                   f"{np.mean([len(it[TRACE_TEXT_FIELD]) for it in items]):.2f}/"
                                   f"{max(len(it[TRACE_TEXT_FIELD]) for it in items)}",
            "len(traces)≠len(verdicts)": sum(len(it[TRACE_TEXT_FIELD]) != len(it[VERDICT_LIST_FIELD]) for it in items),
            "verdict values": dict(vals),
            "UNKNOWN verdict values": {k: c for k, c in vals.items() if k not in label_map},
            "label dist (F/T/C)": [int((labels == c).sum()) for c in range(3)],
            "plurality all: Macro-F1": f1(p_all), "plurality all: Acc": round(accuracy_score(labels, p_all), 4),
            f"plurality top-{TOP_K}: Macro-F1": f1(p_top), f"plurality top-{TOP_K}: Acc": round(accuracy_score(labels, p_top), 4)}

DATA_DIAG = {s: data_diagnostics(s) for s in SPLITS}
display(pd.DataFrame(DATA_DIAG))
for s, dg in DATA_DIAG.items():
    if dg["UNKNOWN verdict values"]:
        print(f"⚠ {s}: verdict values outside label_map are IGNORED by verdict_features: {dg['UNKNOWN verdict values']}")
with open(f"{OUTPUT_DIR}/data_diagnostics.json", "w", encoding="utf-8") as _f:
    json.dump(to_py({"ranking": rank_rows, "data": DATA_DIAG}), _f, indent=2, ensure_ascii=False)

## 7. Feature Engineering
Per row: claim `(D,)`, top-5 traces `(5, D)` **in rank order** (slot 0 = rank 1), mask `(5,)`, verdict `(7,)`, label.

| idx | verdict feature (over the 5 selected traces) | description |
|:-:|:--|:--|
| 0–2 | frac_false / frac_true / frac_conflicting | vote shares |
| 3 | majority_frac | plurality share |
| 4 | distinct_ratio | classes with ≥ 1 vote / 3 |
| 5 | entropy | Shannon entropy of vote shares |
| 6 | num_traces_norm | min(valid votes / 5, 1) |

The embedding arrays are read one row at a time and only the claim and the 5 selected trace rows are kept.
Each array must have `1 + #non-empty traces` rows and unit-norm, finite vectors.

In [ ]:
def verdict_features(verdict_list, max_traces=TOP_K):
    # 7 scalar features from the vote distribution of the selected traces (same formulas as mlp-english-2).
    counts = {0: 0, 1: 0, 2: 0}
    for v in verdict_list:
        if v in label_map:
            counts[label_map[v]] += 1
    total = sum(counts.values())
    if total == 0:
        return np.zeros(SCALAR_DIM, dtype=np.float32)
    frac_false, frac_true, frac_conf = counts[0] / total, counts[1] / total, counts[2] / total
    majority_frac   = max(counts.values()) / total
    distinct_ratio  = sum(1 for c in counts.values() if c > 0) / 3.0
    entropy         = -sum(p * np.log(p + 1e-9) for p in [frac_false, frac_true, frac_conf] if p > 0)
    num_traces_norm = min(total / max_traces, 1.0)
    return np.array([frac_false, frac_true, frac_conf, majority_frac, distinct_ratio, entropy, num_traces_norm],
                    dtype=np.float32)

def build_arrays(split):
    d, N = SPLITS[split], len(SPLITS[split]["order"])
    claims, traces = None, None
    masks   = np.zeros((N, TOP_K), dtype=np.float32)
    scalars = np.empty((N, SCALAR_DIM), dtype=np.float32)
    labels  = np.empty((N,), dtype=np.int64)
    picked  = np.full((N, TOP_K), -1, dtype=np.int64)
    bad_rows, bad_norm = 0, 0
    with np.load(EMB_PATHS[split], allow_pickle=False) as z:
        for r, id_ in enumerate(d["order"]):
            it, key = d["id_to_item"][id_], f"id_{id_}"
            if key not in z.files:
                raise KeyError(f"{split}: {key} missing from {EMB_PATHS[split]} — embeddings do not match this JSON")
            a = z[key]
            if claims is None:
                claims = np.empty((N, a.shape[1]), dtype=np.float32)
                traces = np.zeros((N, TOP_K, a.shape[1]), dtype=np.float32)
            nonempty = [j for j, t in enumerate(it[TRACE_TEXT_FIELD]) if t.strip()]
            bad_rows += a.shape[0] != 1 + len(nonempty)
            row_of = {j: 1 + p for p, j in enumerate(nonempty)}       # trace index → embedding row
            sel = TOPK[split][id_]
            rows = [row_of[j] for j in sel]
            bad_norm += (not np.isfinite(a[[0] + rows]).all()) or abs(np.linalg.norm(a[[0] + rows], axis=1) - 1).max() > 1e-3
            claims[r] = a[0]
            traces[r, :len(rows)] = a[rows]
            masks[r, :len(rows)] = 1.0
            picked[r, :len(sel)] = sel
            scalars[r] = verdict_features([it[VERDICT_LIST_FIELD][j] for j in sel])
            labels[r] = get_label(it)
    assert bad_rows == 0, f"{split}: {bad_rows} rows have an embedding row count ≠ 1 + #non-empty traces — wrong npz?"
    assert bad_norm == 0, f"{split}: {bad_norm} rows have non-finite or non-unit-norm embeddings"
    return {"ids": list(d["order"]), "claim": torch.from_numpy(claims), "traces": torch.from_numpy(traces),
            "mask": torch.from_numpy(masks), "scalars": torch.from_numpy(scalars), "labels": torch.from_numpy(labels),
            "picked": picked}

ARR = {}
for split in SPLITS:
    t = time.time()
    ARR[split] = build_arrays(split)
    print(f"{split:5s}: claim {tuple(ARR[split]['claim'].shape)} | traces {tuple(ARR[split]['traces'].shape)} | {time.time() - t:.0f}s")
    gc.collect()
EMB_DIM = ARR["train"]["claim"].shape[1]

feat_rows = {}
for split, a in ARR.items():
    sc = a["scalars"].numpy(); nt = a["mask"].sum(1).numpy()
    tr = a["traces"]; m = a["mask"]
    feat_rows[split] = {"N": len(a["ids"]), "traces used mean": f"{nt.mean():.2f}",
                        "items with 0 traces": int((nt == 0).sum()),
                        "all-zero verdict rows": int((np.abs(sc).sum(1) == 0).sum()),
                        "mean within-item trace std": f"{float(tr[m.sum(1) > 1].std(dim=1).mean()):.5f}",
                        **{f"mean {n}": f"{sc[:, i].mean():.3f}" for i, n in enumerate(
                           ["fFalse", "fTrue", "fConf", "major", "distinct", "entropy", "nTr"])}}
display(pd.DataFrame(feat_rows))
print(f"✓ Features built | EMB_DIM = {EMB_DIM}")

_cnt = Counter(ARR["train"]["labels"].tolist())
class_weights = torch.tensor([len(ARR["train"]["ids"]) / (3.0 * _cnt[c]) for c in range(3)], dtype=torch.float32)
print("Train label counts:", {reverse_label_map[c]: _cnt[c] for c in range(3)})
print("Class weights     :", [round(w, 4) for w in class_weights.tolist()])

## 8. Model Components
* **RankAwareAttentionPool** (changed): `score_i = Linear(D,256)→Tanh→Linear(256,1)` applied to `trace_i + rank_emb[i]`,
  masked softmax over the 5 traces, output = Σ weight_i · trace_i. `rank_emb` is a learnable `(5, D)` matrix
  initialised to zero, so the module starts as the plain attention of `mlp-english-2` and learns how much the
  ranker's order should matter. A row with no valid trace gets a zero vector instead of NaN.
* **masked_std** (same module): element-wise standard deviation over the 5 valid traces (disagreement).
* **Verdict representations** (7 → 1024): Raw7, Dup_1024, AE_1024 (frozen pre-trained encoder), Proj_1024. Unchanged.
* **Classifier**: `BatchNorm1d(input) → [Linear→BN→ReLU→Dropout(0.3)] × (512, 256, 128) → Linear(3)`. Unchanged.
* Pooling and std are computed in fp32; only the MLP runs under AMP.

In [ ]:
class RankAwareAttentionPool(nn.Module):
    def __init__(self, dim, n_ranks=TOP_K, hidden=ATTN_HIDDEN):
        super().__init__()
        self.rank_emb = nn.Parameter(torch.zeros(n_ranks, dim))      # zero init = plain attention at the start
        self.attn = nn.Sequential(nn.Linear(dim, hidden), nn.Tanh(), nn.Linear(hidden, 1))
    def forward(self, traces, mask):
        scores  = self.attn(traces + self.rank_emb[: traces.shape[1]]).squeeze(-1)
        scores  = scores.masked_fill(mask == 0, float("-inf"))
        has_any = mask.sum(dim=1, keepdim=True) > 0
        scores  = torch.where(has_any, scores, torch.zeros_like(scores))   # avoid softmax(all -inf)=NaN
        weights = torch.softmax(scores, dim=1) * has_any                    # zero vector if no trace
        return torch.bmm(weights.unsqueeze(1), traces).squeeze(1)

def masked_std(traces, mask, eps=1e-6):
    mask_f = mask.unsqueeze(-1)
    count  = mask_f.sum(dim=1).clamp(min=1.0)
    mean   = (traces * mask_f).sum(dim=1) / count
    var    = ((traces - mean.unsqueeze(1)) ** 2 * mask_f).sum(dim=1) / count
    return torch.sqrt(var + eps)

class CyclicDuplication(nn.Module):
    def __init__(self, in_dim=SCALAR_DIM, out_dim=EXPANDED_SCALAR_DIM):
        super().__init__()
        self.out_dim, self.repeats = out_dim, math.ceil(out_dim / in_dim)
    def forward(self, x):
        return x.repeat(1, self.repeats)[:, :self.out_dim]

class LearnableProjection(nn.Module):
    def __init__(self, in_dim=SCALAR_DIM, hidden=PROJ_HIDDEN, out_dim=EXPANDED_SCALAR_DIM):
        super().__init__()
        self.proj = nn.Sequential(nn.Linear(in_dim, hidden), nn.BatchNorm1d(hidden), nn.ReLU(),
                                  nn.Linear(hidden, out_dim), nn.BatchNorm1d(out_dim), nn.ReLU())
    def forward(self, x): return self.proj(x)

class VerdictAutoencoder(nn.Module):
    def __init__(self, in_dim=SCALAR_DIM, hidden=AE_HIDDEN, latent=EXPANDED_SCALAR_DIM):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(in_dim, hidden), nn.ReLU(), nn.Linear(hidden, latent), nn.ReLU())
        self.decoder = nn.Sequential(nn.Linear(latent, hidden), nn.ReLU(), nn.Linear(hidden, in_dim))
    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z), z

class FrozenAEEncoder(nn.Module):
    # Own deep copy of the pre-trained encoder; never updated, always in eval mode.
    def __init__(self, ae):
        super().__init__()
        self.encoder = copy.deepcopy(ae.encoder)
        for p in self.encoder.parameters(): p.requires_grad_(False)
    def train(self, mode=True): return super().train(False)
    def forward(self, x): return self.encoder(x)

VERDICT_DIMS = {"raw7": SCALAR_DIM, "dup": EXPANDED_SCALAR_DIM, "ae": EXPANDED_SCALAR_DIM, "proj": EXPANDED_SCALAR_DIM}

class AblationMLP(nn.Module):
    def __init__(self, cfg, emb_dim, ae=None):
        super().__init__()
        self.use_attention, self.use_disagreement, vm = cfg["attention"], cfg["disagreement"], cfg["verdict"]
        if self.use_attention: self.attn_pool = RankAwareAttentionPool(emb_dim)
        if   vm == "raw7": self.verdict_repr = nn.Identity()
        elif vm == "dup":  self.verdict_repr = CyclicDuplication()
        elif vm == "ae":
            if ae is None: raise ValueError("AE verdict mode needs the pre-trained autoencoder")
            self.verdict_repr = FrozenAEEncoder(ae)
        elif vm == "proj": self.verdict_repr = LearnableProjection()
        else: raise ValueError(vm)
        self.input_dim = emb_dim * (1 + self.use_attention + self.use_disagreement) + VERDICT_DIMS[vm]
        layers, prev = [nn.BatchNorm1d(self.input_dim)], self.input_dim
        for h in HIDDEN_DIMS:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(DROPOUT)]
            prev = h
        layers.append(nn.Linear(prev, OUTPUT_DIM))
        self.mlp = nn.Sequential(*layers)

    def features(self, claim, traces, mask, scalars):
        with torch.autocast(device_type=claim.device.type, enabled=False):
            claim, traces, mask, scalars = claim.float(), traces.float(), mask.float(), scalars.float()
            parts = [claim]                                                    # x_claim
            if self.use_attention:    parts.append(self.attn_pool(traces, mask))  # x_attention (rank-aware)
            if self.use_disagreement: parts.append(masked_std(traces, mask))      # x_disagreement
            parts.append(self.verdict_repr(scalars))                           # x_verdict
            return torch.cat(parts, dim=1)

    def forward(self, claim, traces, mask, scalars):
        return self.mlp(self.features(claim, traces, mask, scalars))

## 9. Autoencoder Pre-training (AE_1024)
Pre-trained on the **train** top-5 verdict features; epoch selected by validation reconstruction MSE (no labels used).

In [ ]:
def pretrain_autoencoder():
    path = f"{MODELS_DIR}/verdict_autoencoder.pt"
    set_seed()
    ae = VerdictAutoencoder().to(DEVICE)
    if os.path.exists(path) and not FORCE_RERUN_EXPERIMENTS:
        ae.load_state_dict(torch.load(path, map_location=DEVICE)); ae.eval()
        print(f"Loaded pre-trained AE ← {path}")
    else:
        Xtr, Xva = ARR["train"]["scalars"].to(DEVICE), ARR["val"]["scalars"].to(DEVICE)
        opt, mse = optim.Adam(ae.parameters(), lr=AE_LR), nn.MSELoss()
        g = torch.Generator().manual_seed(RANDOM_SEED)
        loader = DataLoader(TensorDataset(ARR["train"]["scalars"]), batch_size=AE_BATCH_SIZE, shuffle=True, generator=g)
        best, best_state = float("inf"), None
        for ep in range(AE_EPOCHS):
            ae.train()
            for (xb,) in loader:
                xb = xb.to(DEVICE); opt.zero_grad()
                loss = mse(ae(xb)[0], xb); loss.backward(); opt.step()
            ae.eval()
            with torch.no_grad():
                tr, va = mse(ae(Xtr)[0], Xtr).item(), mse(ae(Xva)[0], Xva).item()
            if va < best: best, best_state = va, copy.deepcopy(ae.state_dict())
            if (ep + 1) % 25 == 0: print(f"  AE epoch {ep+1:3d}/{AE_EPOCHS}  train MSE {tr:.2e}  val MSE {va:.2e}")
        ae.load_state_dict(best_state); ae.eval()
        torch.save(ae.state_dict(), path)
        print(f"Best val reconstruction MSE {best:.2e} → saved {path}")
    with torch.no_grad():
        for s in SPLITS:
            X = ARR[s]["scalars"].to(DEVICE); r, z = ae(X)
            print(f"  {s:<5} recon MSE {F.mse_loss(r, X).item():.2e} | latent {tuple(z.shape)} | "
                  f"active dims {(z > 0).float().mean().item():.3f}")
    return ae.cpu()

PRETRAINED_AE = pretrain_autoencoder()

## 10. Training & Evaluation
* Loss: `CrossEntropyLoss(weight = N / (3·n_c))` on train. Adam (lr 1e-3, wd 1e-5), ReduceLROnPlateau (0.5, patience 3), gradient clip 1.0, AMP, DataParallel on 2 GPUs.
* The checkpoint kept is the one with the lowest **validation loss**, with early-stopping patience 20.
* Metrics per split: confusion matrix, **Macro-F1**, **Accuracy**, per-class P/R/F1.
* Saved per experiment: `results/<exp>.json`, `models/<exp>.pt`, `predictions/<exp>_<split>.csv` (probabilities,
  prediction and the 5 trace indices that were fed), confusion-matrix PNGs.

In [ ]:
def make_loader(split, shuffle):
    a = ARR[split]
    ds = TensorDataset(a["claim"], a["traces"], a["mask"], a["scalars"], a["labels"])
    kw = dict(batch_size=BATCH_SIZE, num_workers=0, pin_memory=torch.cuda.is_available())
    if shuffle:
        return DataLoader(ds, shuffle=True, drop_last=True,          # drop_last: no size-1 BN batch
                          generator=torch.Generator().manual_seed(RANDOM_SEED), **kw)
    return DataLoader(ds, shuffle=False, **kw)

def _to(batch):
    return [b.to(DEVICE, non_blocking=True) for b in batch]

def train_epoch(model, loader, optimizer, criterion, scaler):
    model.train(); total = 0.0
    for claim, traces, mask, scalars, y in loader:
        claim, traces, mask, scalars, y = _to((claim, traces, mask, scalars, y))
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=scaler is not None):
            loss = criterion(model(claim, traces, mask, scalars), y)
        if not torch.isfinite(loss):
            raise RuntimeError("Non-finite training loss — aborting experiment")
        if scaler is not None:
            scaler.scale(loss).backward(); scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            scaler.step(optimizer); scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            optimizer.step()
        total += loss.item()
    return total / len(loader)

@torch.no_grad()
def evaluate(model, loader, criterion):
    # Returns the exact class-weighted mean CE over the whole split (same definition as the training loss).
    model.eval(); loss_sum, w_sum, probs, labels = 0.0, 0.0, [], []
    w = criterion.weight
    for claim, traces, mask, scalars, y in loader:
        claim, traces, mask, scalars, y = _to((claim, traces, mask, scalars, y))
        logits = model(claim, traces, mask, scalars).float()
        loss_sum += F.cross_entropy(logits, y, weight=w, reduction="sum").item()
        w_sum    += (w[y].sum() if w is not None else torch.tensor(float(len(y)))).item()
        probs.append(torch.softmax(logits, dim=1).cpu()); labels.append(y.cpu())
    probs, labels = torch.cat(probs).numpy(), torch.cat(labels).numpy()
    return loss_sum / w_sum, probs, labels

def compute_metrics(y, pred):
    p = precision_score(y, pred, labels=[0,1,2], average=None, zero_division=0)
    r = recall_score(y, pred, labels=[0,1,2], average=None, zero_division=0)
    f = f1_score(y, pred, labels=[0,1,2], average=None, zero_division=0)
    return {"confusion_matrix": confusion_matrix(y, pred, labels=[0,1,2]).tolist(),
            "accuracy": float(accuracy_score(y, pred)),
            "macro_f1": float(f.mean()), "macro_precision": float(p.mean()), "macro_recall": float(r.mean()),
            "weighted_f1": float(f1_score(y, pred, labels=[0,1,2], average="weighted", zero_division=0)),
            "per_class": {CLASS_NAMES[c]: {"precision": float(p[c]), "recall": float(r[c]), "f1": float(f[c])}
                          for c in range(3)}}

def print_report(name, split, m):
    print(f"\n── {name} | {split.upper()} ──  Macro-F1 {m['macro_f1']:.4f} | Accuracy {m['accuracy']:.4f}")
    print("Confusion matrix (rows=true, cols=pred; False/True/Conflicting):")
    print(np.array(m["confusion_matrix"]))
    for c in CLASS_NAMES:
        pc = m["per_class"][c]
        print(f"  {c:<12} P {pc['precision']:.4f}  R {pc['recall']:.4f}  F1 {pc['f1']:.4f}")

def save_cm_plot(cm, title, path):
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(np.array(cm), annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(title, fontsize=10)
    fig.tight_layout(); fig.savefig(path, dpi=110); plt.close(fig)

In [ ]:
def run_experiment(exp):
    name, cfg = exp["name"], exp["cfg"]
    res_path = f"{RESULTS_DIR}/{name}.json"
    if os.path.exists(res_path) and not FORCE_RERUN_EXPERIMENTS:
        print(f"✓ {name}: result exists — loaded ({res_path})")
        with open(res_path, encoding="utf-8") as f: return json.load(f)

    print(f"\n{'#'*78}\n# {exp['display']}   [{name}]\n# cfg = {cfg}\n{'#'*78}")
    set_seed()
    base = AblationMLP(cfg, EMB_DIM, ae=PRETRAINED_AE).to(DEVICE)
    n_total = sum(p.numel() for p in base.parameters())
    n_train = sum(p.numel() for p in base.parameters() if p.requires_grad)
    print(f"input_dim = {base.input_dim} | params total {n_total:,} | trainable {n_train:,}")
    model = nn.DataParallel(base) if N_GPUS > 1 else base

    optimizer = optim.Adam([p for p in base.parameters() if p.requires_grad], lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    criterion = nn.CrossEntropyLoss(weight=class_weights.to(DEVICE))
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=LR_PLATEAU_FACTOR,
                                                     patience=LR_PLATEAU_PATIENCE)
    scaler = torch.amp.GradScaler("cuda") if (USE_AMP and DEVICE == "cuda") else None
    train_loader = make_loader("train", True)
    eval_loaders = {s: make_loader(s, False) for s in SPLITS}

    best_val, best_state, best_epoch, patience = float("inf"), None, -1, 0
    hist = {"train_loss": [], "val_loss": [], "val_macro_f1": [], "lr": []}
    t0 = time.time()
    for ep in range(1, MAX_EPOCHS + 1):
        tr_loss = train_epoch(model, train_loader, optimizer, criterion, scaler)
        va_loss, va_probs, va_y = evaluate(model, eval_loaders["val"], criterion)
        va_f1 = f1_score(va_y, va_probs.argmax(1), labels=[0,1,2], average="macro", zero_division=0)
        hist["train_loss"].append(tr_loss); hist["val_loss"].append(va_loss)
        hist["val_macro_f1"].append(float(va_f1)); hist["lr"].append(optimizer.param_groups[0]["lr"])
        scheduler.step(va_loss)
        flag = ""
        if va_loss < best_val:
            best_val, best_epoch, patience, flag = va_loss, ep, 0, "  ← best"
            best_state = copy.deepcopy(base.state_dict())
        else:
            patience += 1
        print(f"  ep {ep:3d} | train {tr_loss:.4f} | val {va_loss:.4f} | val MacroF1 {va_f1:.4f} | "
              f"lr {hist['lr'][-1]:.1e}{flag}")
        if patience >= EARLY_STOP_PATIENCE:
            print(f"  early stop at epoch {ep}"); break
    if best_state is None:
        raise RuntimeError(f"{name}: no valid checkpoint (validation loss never finite)")
    train_time = time.time() - t0

    base.load_state_dict(best_state)
    torch.save(best_state, f"{MODELS_DIR}/{name}.pt")
    metrics = {}
    for s, ld in eval_loaders.items():
        loss, probs, y = evaluate(model, ld, criterion)
        pred = probs.argmax(1)
        metrics[s] = {"loss": float(loss), **compute_metrics(y, pred)}
        print_report(exp["display"], s, metrics[s])
        save_cm_plot(metrics[s]["confusion_matrix"], f"{exp['display']} — {s}", f"{PLOTS_DIR}/cm_{name}_{s}.png")
        pd.DataFrame({"id": ARR[s]["ids"], "true_label": [reverse_label_map[int(v)] for v in y],
                      "predicted_verdict": [reverse_label_map[int(v)] for v in pred],
                      "prob_False": probs[:, 0], "prob_True": probs[:, 1], "prob_Conflicting": probs[:, 2],
                      f"top{TOP_K}_trace_indices": [" ".join(str(j) for j in p if j >= 0) for p in ARR[s]["picked"]]}
                     ).to_csv(f"{PREDS_DIR}/{name}_{s}.csv", index=False)

    attn_rank = None
    if cfg["attention"]:
        attn_rank = base.attn_pool.rank_emb.detach().norm(dim=1).cpu().numpy().round(4).tolist()
        print(f"  learned rank-embedding norms (rank 1..{TOP_K}): {attn_rank}")
    result = {"name": name, "display": exp["display"], "group": exp["group"], "cfg": cfg,
              "input_dim": base.input_dim, "params_total": n_total, "params_trainable": n_train,
              "best_epoch": best_epoch, "epochs_run": len(hist["train_loss"]), "train_time_s": train_time,
              "rank_embedding_norms": attn_rank, "history": hist, "metrics": metrics}
    with open(res_path, "w", encoding="utf-8") as f: json.dump(to_py(result), f, indent=2)
    del model, base, best_state; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return result

## 11. Ablation Configurations

In [ ]:
def _exp(idx, display, group, attention, disagreement, verdict):
    return {"name": f"{idx:02d}_{display.replace(' + ', '__').replace(' ', '_')}", "display": display,
            "group": group, "cfg": {"attention": attention, "disagreement": disagreement, "verdict": verdict}}

EXPERIMENTS = [
    _exp(1,  "Full + Raw7",                        "Full",            True,  True,  "raw7"),
    _exp(2,  "Full + Duplication_1024",            "Full",            True,  True,  "dup"),
    _exp(3,  "Full + AE_1024",                     "Full",            True,  True,  "ae"),
    _exp(4,  "Full + Projection_1024",             "Full",            True,  True,  "proj"),
    _exp(5,  "No_Attention + Duplication_1024",    "No_Attention",    False, True,  "dup"),
    _exp(6,  "No_Attention + AE_1024",             "No_Attention",    False, True,  "ae"),
    _exp(7,  "No_Attention + Projection_1024",     "No_Attention",    False, True,  "proj"),
    _exp(8,  "No_Disagreement + Duplication_1024", "No_Disagreement", True,  False, "dup"),
    _exp(9,  "No_Disagreement + AE_1024",          "No_Disagreement", True,  False, "ae"),
    _exp(10, "No_Disagreement + Projection_1024",  "No_Disagreement", True,  False, "proj"),
]
VERDICT_NAMES = {"raw7": "Raw 7", "dup": "Dup_1024", "ae": "AE_1024", "proj": "Proj_1024"}

# Shape / NaN check of all 10 models on a real train batch (including a forced 0-trace row)
_b = next(iter(make_loader("train", False)))
_c, _t, _m, _s, _ = [x[:4].clone().to(DEVICE) for x in _b]
_m[0] = 0; _t[0] = 0
rows = []
for e in EXPERIMENTS:
    _net = AblationMLP(e["cfg"], EMB_DIM, ae=PRETRAINED_AE).to(DEVICE).eval()
    with torch.no_grad():
        x, out = _net.features(_c, _t, _m, _s), _net(_c, _t, _m, _s)
    assert x.shape == (4, _net.input_dim) and out.shape == (4, OUTPUT_DIM) and torch.isfinite(out).all()
    rows.append({"#": e["name"][:2], "Configuration": e["display"], "input_dim": _net.input_dim,
                 "trainable params": sum(p.numel() for p in _net.parameters() if p.requires_grad)})
del _net, _b, _c, _t, _m, _s; gc.collect()
display(pd.DataFrame(rows))
print("✓ All 10 configurations build and produce finite outputs")

## 12. Run All 10 MLP Experiments

In [ ]:
ALL_RESULTS = [run_experiment(e) for e in EXPERIMENTS]
print(f"\n✓ {len(ALL_RESULTS)}/10 experiments complete")

## 13. Results
Main table. Selection is on validation loss; **test** columns are the reported numbers.

In [ ]:
def results_table(results):
    rows = []
    for r in results:
        v, t = r["metrics"]["val"], r["metrics"]["test"]
        rows.append({"#": int(r["name"][:2]), "Configuration": r["display"], "Group": r["group"],
                     "Verdict": VERDICT_NAMES[r["cfg"]["verdict"]], "Input dim": r["input_dim"],
                     "Trainable params": r["params_trainable"], "Best ep": r["best_epoch"], "Epochs": r["epochs_run"],
                     "Val Macro-F1": v["macro_f1"], "Val Acc": v["accuracy"],
                     "Test Macro-F1": t["macro_f1"], "Test Acc": t["accuracy"],
                     "Test F1 False": t["per_class"]["False"]["f1"], "Test F1 True": t["per_class"]["True"]["f1"],
                     "Test F1 Conflicting": t["per_class"]["Conflicting"]["f1"],
                     "Train time (s)": round(r["train_time_s"], 1)})
    return pd.DataFrame(rows).sort_values("#").reset_index(drop=True)

RES = results_table(ALL_RESULTS)
RES.to_csv(f"{OUTPUT_DIR}/mlp_ablation_results.csv", index=False)
with open(f"{OUTPUT_DIR}/mlp_ablation_results_full.json", "w", encoding="utf-8") as f: json.dump(to_py(ALL_RESULTS), f, indent=2)
_fmt = {c: "{:.4f}" for c in RES.columns if "F1" in c or "Acc" in c}
display(RES.style.format(_fmt)
           .highlight_max(subset=["Val Macro-F1", "Test Macro-F1", "Test Acc"], color="#c8e6c9"))
bt = RES.loc[RES["Test Macro-F1"].idxmax()]; bv = RES.loc[RES["Val Macro-F1"].idxmax()]
print(f"Best by TEST Macro-F1: {bt['Configuration']}  ({bt['Test Macro-F1']:.4f}, Acc {bt['Test Acc']:.4f})")
print(f"Best by VAL  Macro-F1: {bv['Configuration']}  ({bv['Val Macro-F1']:.4f}) → its test Macro-F1 {bv['Test Macro-F1']:.4f}")

## 14. Ablation Comparisons
1. **Configuration × verdict-representation matrices** (Macro-F1 and Accuracy, val and test)
2. **Effect of verdict expansion**: Full + X − Full + Raw7
3. **Effect of removing a component**: No_Attention + X − Full + X and No_Disagreement + X − Full + X

In [ ]:
def matrix(metric):
    m = RES.pivot(index="Group", columns="Verdict", values=metric)
    return m.reindex(index=["Full", "No_Attention", "No_Disagreement"],
                     columns=["Raw 7", "Dup_1024", "AE_1024", "Proj_1024"])
MATS = {k: matrix(k) for k in ["Test Macro-F1", "Test Acc", "Val Macro-F1", "Val Acc"]}
for k, m in MATS.items():
    print(f"\n{k}  (rows = feature configuration, cols = verdict representation; '—' = not in the ablation)")
    display(m.style.format("{:.4f}", na_rep="—").highlight_max(axis=None, color="#c8e6c9"))
    m.to_csv(f"{OUTPUT_DIR}/matrix_{k.replace(' ', '_').replace('-', '')}.csv")

get = lambda g, v, col: float(RES[(RES.Group == g) & (RES.Verdict == v)][col].iloc[0])
EXPANSION_DELTA = pd.DataFrame([{"Verdict repr (Full)": v,
                                 "Δ Test Macro-F1": get("Full", v, "Test Macro-F1") - get("Full", "Raw 7", "Test Macro-F1"),
                                 "Δ Test Acc":      get("Full", v, "Test Acc")      - get("Full", "Raw 7", "Test Acc"),
                                 "Δ Val Macro-F1":  get("Full", v, "Val Macro-F1")  - get("Full", "Raw 7", "Val Macro-F1")}
                                for v in ["Dup_1024", "AE_1024", "Proj_1024"]])
print("\nEffect of verdict expansion vs Full + Raw7:")
display(EXPANSION_DELTA.style.format({c: "{:+.4f}" for c in EXPANSION_DELTA.columns[1:]}))

rem_rows = []
for g in ["No_Attention", "No_Disagreement"]:
    for v in ["Dup_1024", "AE_1024", "Proj_1024"]:
        rem_rows.append({"Removed": g.replace("No_", ""), "Verdict": v,
                         "Δ Test Macro-F1": get(g, v, "Test Macro-F1") - get("Full", v, "Test Macro-F1"),
                         "Δ Test Acc":      get(g, v, "Test Acc")      - get("Full", v, "Test Acc"),
                         "Δ Test F1 Conflicting": get(g, v, "Test F1 Conflicting") - get("Full", v, "Test F1 Conflicting"),
                         "Δ Val Macro-F1":  get(g, v, "Val Macro-F1")  - get("Full", v, "Val Macro-F1")})
REMOVAL_DELTA = pd.DataFrame(rem_rows)
print("\nEffect of removing a component (vs Full with the same verdict representation; negative = component helps):")
display(REMOVAL_DELTA.style.format({c: "{:+.4f}" for c in REMOVAL_DELTA.columns[2:]}))
print("\nMean over verdict representations:")
display(REMOVAL_DELTA.groupby("Removed")[REMOVAL_DELTA.columns[2:]].mean().style.format("{:+.4f}"))
EXPANSION_DELTA.to_csv(f"{OUTPUT_DIR}/delta_verdict_expansion.csv", index=False)
REMOVAL_DELTA.to_csv(f"{OUTPUT_DIR}/delta_component_removal.csv", index=False)

## 15. Plots

In [ ]:
TITLE = f"MLP ablation, top-{TOP_K} ranked traces ({RANK_METHOD}), Qwen3-Embedding-8B (English)"

# (a) Macro-F1 and Accuracy per configuration
fig, axes = plt.subplots(1, 2, figsize=(17, 6))
x = np.arange(len(RES)); w = 0.38
for ax, metric in zip(axes, ["Macro-F1", "Acc"]):
    ax.bar(x - w/2, RES[f"Val {metric}"],  w, label="Validation", color="#90caf9")
    b2 = ax.bar(x + w/2, RES[f"Test {metric}"], w, label="Test", color="#1565c0")
    ax.bar_label(b2, fmt="%.3f", fontsize=7, padding=2)
    ax.set_xticks(x); ax.set_xticklabels([f"{i}. {c}" for i, c in zip(RES["#"], RES["Configuration"])],
                                         rotation=55, ha="right", fontsize=8)
    ax.set_ylim(0, 1); ax.set_title(f"{metric} per configuration", fontweight="bold"); ax.legend()
    ax.spines[["top", "right"]].set_visible(False)
fig.suptitle(TITLE, fontweight="bold"); fig.tight_layout()
fig.savefig(f"{PLOTS_DIR}/ablation_bars.png", dpi=130, bbox_inches="tight"); plt.show()

# (b) Matrix heatmaps
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
for ax, k in zip(axes, ["Test Macro-F1", "Test Acc"]):
    sns.heatmap(MATS[k], annot=True, fmt=".4f", cmap="YlGn", cbar=False, ax=ax, linewidths=.5)
    ax.set_title(k, fontweight="bold"); ax.set_xlabel("Verdict representation"); ax.set_ylabel("")
fig.tight_layout(); fig.savefig(f"{PLOTS_DIR}/ablation_matrix.png", dpi=130, bbox_inches="tight"); plt.show()

# (c) Training curves
fig, axes = plt.subplots(2, 5, figsize=(22, 8)); axes = axes.ravel()
for ax, r in zip(axes, sorted(ALL_RESULTS, key=lambda r: r["name"])):
    h = r["history"]; ep = range(1, len(h["train_loss"]) + 1)
    ax.plot(ep, h["train_loss"], label="train loss"); ax.plot(ep, h["val_loss"], label="val loss")
    ax2 = ax.twinx(); ax2.plot(ep, h["val_macro_f1"], color="green", alpha=.6, label="val Macro-F1")
    ax2.set_ylim(0, 1); ax.axvline(r["best_epoch"], color="red", ls="--", lw=1)
    ax.set_title(r["display"], fontsize=9, fontweight="bold"); ax.set_xlabel("epoch")
axes[0].legend(fontsize=7, loc="upper right")
fig.suptitle("Loss curves (red line = selected checkpoint)", fontweight="bold"); fig.tight_layout()
fig.savefig(f"{PLOTS_DIR}/loss_curves.png", dpi=110, bbox_inches="tight"); plt.show()

# (d) Test confusion matrices
fig, axes = plt.subplots(2, 5, figsize=(24, 9)); axes = axes.ravel()
for ax, r in zip(axes, sorted(ALL_RESULTS, key=lambda r: r["name"])):
    m = r["metrics"]["test"]
    sns.heatmap(np.array(m["confusion_matrix"]), annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    ax.set_title(f"{r['display']}\nMacro-F1 {m['macro_f1']:.4f} | Acc {m['accuracy']:.4f}", fontsize=9)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
fig.suptitle("Test confusion matrices", fontweight="bold"); fig.tight_layout()
fig.savefig(f"{PLOTS_DIR}/test_confusion_matrices.png", dpi=110, bbox_inches="tight"); plt.show()

## 16. Paper Table (LaTeX) & Output Manifest

In [ ]:
best_t, best_a = RES["Test Macro-F1"].max(), RES["Test Acc"].max()
fmt = lambda v, b: (r"\textbf{%.4f}" % v) if abs(v - b) < 1e-12 else "%.4f" % v
lines = [r"\begin{table}[t]", r"\centering",
         r"\caption{MLP ablation on the English test set using the top-%d traces ranked by our p/n ranker "
         r"(Qwen3-Embedding-8B, seed 42).}" % TOP_K,
         r"\label{tab:mlp-ablation-top%d}" % TOP_K, r"\begin{tabular}{rlrcc}", r"\hline",
         r"\# & Configuration & Dim & Macro-F1 & Acc. \\", r"\hline"]
for _, r in RES.iterrows():
    cfg = r["Configuration"].replace("_", r"\_")
    lines.append(f"{r['#']} & {cfg} & {r['Input dim']} & {fmt(r['Test Macro-F1'], best_t)} & "
                 f"{fmt(r['Test Acc'], best_a)} \\\\")
lines += [r"\hline", r"\end{tabular}", r"\end{table}"]
with open(f"{OUTPUT_DIR}/mlp_ablation_table.tex", "w", encoding="utf-8") as f: f.write("\n".join(lines))
print("\n".join(lines))

print("\n── Output files ──")
for root, _, files in os.walk(OUTPUT_DIR):
    if "/." in root: continue
    for fn in sorted(files):
        p = os.path.join(root, fn)
        print(f"  {os.path.relpath(p, OUTPUT_DIR):<70} {os.path.getsize(p) / 1e6:>10.2f} MB")

## 17. Notes
* **Leakage:** the train/val top-5 come from out-of-fold rankings, and test from the ranker fitted on train+val;
  no gold test label is used before evaluation. The feature-block selection of the ranker used 5-fold CV over
  train+val, so train/val rankings are very slightly optimistic; this does not affect test.
* **Verdict features** now describe the 5 selected traces. Under the p/n rule the top 5 are chosen by group
  (`p_c / n_c`), so their composition reflects the ranker's label estimate. This is a learned, out-of-fold signal.
* **Rank embedding:** the per-rank norms printed after each run (and stored in `results/<exp>.json`) show how much
  the model used the ranking order; all zeros would mean it ignored it.
* Compare with `mlp-english-2.ipynb` (all 20 traces) for the effect of feeding only the top 5.